## DS605: Fundamentals of Machine Learning Lab Assignment - 5

### Name: Puranik Aryan
### studentid : 202618030

# Garment Employee Productivity — Part A: Scikit-learn Implementation

**Objective:** Build a regression model (Linear Regression) to predict actual_productivity, and a classification model (Logistic Regression) to predict MeetsTarget (whether actual productivity met or exceeded the targeted productivity), using Scikit-learn end-to-end (preprocessing, training, evaluation).

**Dataset:** UCI *Garment Employee Productivity* dataset garments_worker_productivity.csv


## 1. Import Libraries


In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42


## 2. Load the Dataset


In [2]:

df = pd.read_csv("garments_worker_productivity.csv")
print('Shape:', df.shape)
df.head()


Shape: (1197, 15)


,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382


## 3. Initial Data Inspection

Check data types and missing values before doing any preprocessing.


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: float64(6), 

In [4]:
print('Missing values per column:')
df.isnull().sum().sort_values(ascending=False)


Missing values per column:


wip                      506
quarter                    0
date                       0
day                        0
team                       0
targeted_productivity      0
department                 0
smv                        0
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64

## 4. Basic Data Cleaning

In [5]:
df_clean = df.copy()

# Strip whitespace from all object/string columns
obj_cols = df_clean.select_dtypes(include='object').columns
for c in obj_cols:
    df_clean[c] = df_clean[c].astype(str).str.strip().str.lower()

# Fix known typo in 'department'
df_clean['department'] = df_clean['department'].replace({'sweing': 'sewing', 'finishing ': 'finishing'})

# Drop the raw date column if present
if 'date' in df_clean.columns:
    df_clean = df_clean.drop(columns=['date'])

df_clean['department'].value_counts()


C:\Users\phitesh\AppData\Local\Temp\ipykernel_19212\89241404.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df_clean.select_dtypes(include='object').columns


department
sewing       691
finishing    506
Name: count, dtype: int64

## 5. Create the Classification Target

MeetsTarget = 1 when actual_productivity >= targeted_productivity, else 0.


In [6]:
df_clean['MeetsTarget'] = (df_clean['actual_productivity'] >= df_clean['targeted_productivity']).astype(int)
df_clean['MeetsTarget'].value_counts(normalize=True)


MeetsTarget
1    0.730994
0    0.269006
Name: proportion, dtype: float64

## 6. Define Features and Targets

- **Regression:** predict  actual_productivity  from all other features.
- **Classification:** predict  MeetsTarget  from all features **except**  actual_productivity  (to avoid leakage) and  MeetsTarget  itself.


In [7]:
reg_feature_cols = [c for c in df_clean.columns if c not in ['actual_productivity', 'MeetsTarget']]
clf_feature_cols = [c for c in df_clean.columns if c not in ['actual_productivity', 'MeetsTarget']]

X_reg_full = df_clean[reg_feature_cols]
y_reg_full = df_clean['actual_productivity']

X_clf_full = df_clean[clf_feature_cols]
y_clf_full = df_clean['MeetsTarget']

categorical_cols = X_reg_full.select_dtypes(include='object').columns.tolist()
numeric_cols = X_reg_full.select_dtypes(exclude='object').columns.tolist()

print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)


Numeric columns: ['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']
Categorical columns: ['quarter', 'department', 'day']


C:\Users\phitesh\AppData\Local\Temp\ipykernel_19212\3686023420.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_reg_full.select_dtypes(include='object').columns.tolist()


## 7. Train-Test Split (Single Fixed Split)

A single fixed split (same row indices,  random_state=42 ) is used for **both** the regression and classification tasks so results are directly comparable.


In [8]:
train_idx, test_idx = train_test_split(
    df_clean.index, test_size=0.2, random_state=RANDOM_STATE
)

X_reg_train, X_reg_test = X_reg_full.loc[train_idx], X_reg_full.loc[test_idx]
y_reg_train, y_reg_test = y_reg_full.loc[train_idx], y_reg_full.loc[test_idx]

X_clf_train, X_clf_test = X_clf_full.loc[train_idx], X_clf_full.loc[test_idx]
y_clf_train, y_clf_test = y_clf_full.loc[train_idx], y_clf_full.loc[test_idx]

print('Train size:', len(train_idx), '| Test size:', len(test_idx))


Train size: 957 | Test size: 240


## 8. Preprocessing Pipelines

Using  ColumnTransformer  to handle numeric and categorical columns consistently:
- **Numeric:** median imputation + standard scaling.
- **Categorical:** most-frequent imputation + one-hot encoding.

The same preprocessing structure is reused for both the regression and classification pipelines (fit separately on each task's training data).


In [9]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])


## 9. Train Linear Regression (Scikit-learn)

Preprocessing + model are combined into one pipeline. Training and prediction time are recorded.


In [10]:
reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

start = time.perf_counter()
reg_pipeline.fit(X_reg_train, y_reg_train)
sklearn_reg_train_time = time.perf_counter() - start

start = time.perf_counter()
y_reg_pred = reg_pipeline.predict(X_reg_test)
sklearn_reg_predict_time = time.perf_counter() - start

print(f'Training time:   {sklearn_reg_train_time:.6f} s')
print(f'Prediction time: {sklearn_reg_predict_time:.6f} s')


Training time:   0.033108 s
Prediction time: 0.007800 s


## 10. Evaluate the Regression Model


In [11]:
sklearn_mae = mean_absolute_error(y_reg_test, y_reg_pred)
sklearn_rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))  # RMSE (works across sklearn versions)
sklearn_r2 = r2_score(y_reg_test, y_reg_pred)

print(f'MAE:  {sklearn_mae:.4f}')
print(f'RMSE: {sklearn_rmse:.4f}')
print(f'R2:   {sklearn_r2:.4f}')


MAE:  0.1088
RMSE: 0.1481
R2:   0.1736


## 11. Train Logistic Regression (Scikit-learn)

Same pattern as above: a preprocessing + model pipeline, with training and prediction time recorded.


In [12]:
clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

start = time.perf_counter()
clf_pipeline.fit(X_clf_train, y_clf_train)
sklearn_clf_train_time = time.perf_counter() - start

start = time.perf_counter()
y_clf_pred = clf_pipeline.predict(X_clf_test)
sklearn_clf_predict_time = time.perf_counter() - start

print(f'Training time:   {sklearn_clf_train_time:.6f} s')
print(f'Prediction time: {sklearn_clf_predict_time:.6f} s')


Training time:   0.030487 s
Prediction time: 0.008318 s


## 12. Evaluate the Classification Model


In [13]:
sklearn_accuracy = accuracy_score(y_clf_test, y_clf_pred)
sklearn_precision = precision_score(y_clf_test, y_clf_pred, zero_division=0)
sklearn_recall = recall_score(y_clf_test, y_clf_pred, zero_division=0)
sklearn_f1 = f1_score(y_clf_test, y_clf_pred, zero_division=0)

print(f'Accuracy:  {sklearn_accuracy:.4f}')
print(f'Precision: {sklearn_precision:.4f}')
print(f'Recall:    {sklearn_recall:.4f}')
print(f'F1-score:  {sklearn_f1:.4f}')


Accuracy:  0.7500
Precision: 0.7696
Recall:    0.9435
F1-score:  0.8477


## 13. Summary of Part A Results

These values are kept in named variables ( sklearn_* ) so they can be reused for the Scikit-learn vs. manual-NumPy/Pandas comparison in Part B.


In [14]:
summary_part_a = pd.DataFrame({
    'Task': ['Regression (Linear Regression)', 'Classification (Logistic Regression)'],
    'Train Time (s)': [sklearn_reg_train_time, sklearn_clf_train_time],
    'Predict Time (s)': [sklearn_reg_predict_time, sklearn_clf_predict_time],
    'Metric 1': [f'MAE = {sklearn_mae:.4f}', f'Accuracy = {sklearn_accuracy:.4f}'],
    'Metric 2': [f'RMSE = {sklearn_rmse:.4f}', f'Precision = {sklearn_precision:.4f}'],
    'Metric 3': [f'R2 = {sklearn_r2:.4f}', f'Recall = {sklearn_recall:.4f}'],
    'Metric 4': ['-', f'F1 = {sklearn_f1:.4f}'],
})
summary_part_a


,Task,Train Time (s),Predict Time (s),Metric 1,Metric 2,Metric 3,Metric 4
0,Regression (Linear Regression),0.033108,0.007800,MAE = 0.1088,RMSE = 0.1481,R2 = 0.1736,-
1,Classification (Logistic Regression),0.030487,0.008318,Accuracy = 0.7500,Precision = 0.7696,Recall = 0.9435,F1 = 0.8477


# Part B — From-Scratch Implementation (NumPy & Pandas only)

Everything below rebuilds the same workflow (missing-value handling, categorical encoding, scaling, Linear Regression, Logistic Regression, prediction, evaluation) using **only NumPy and Pandas**. No Scikit-learn preprocessing/model/metric utilities are used in this section — only Python's  time  module for timing, and the  train_idx  /  test_idx  computed in Part A so the samples match exactly.


## 14. Rebuild the Raw Train/Test Feature Frames

Same rows ( train_idx ,  test_idx ) and same columns ( numeric_cols ,  categorical_cols ) as Part A, but kept as plain Pandas objects to be processed manually below.


In [15]:
raw_train_df = df_clean.loc[train_idx, numeric_cols + categorical_cols].copy()
raw_test_df  = df_clean.loc[test_idx,  numeric_cols + categorical_cols].copy()

y_reg_train_manual = df_clean.loc[train_idx, 'actual_productivity'].to_numpy().reshape(-1, 1)
y_reg_test_manual  = df_clean.loc[test_idx,  'actual_productivity'].to_numpy().reshape(-1, 1)

y_clf_train_manual = df_clean.loc[train_idx, 'MeetsTarget'].to_numpy().reshape(-1, 1).astype(float)
y_clf_test_manual  = df_clean.loc[test_idx,  'MeetsTarget'].to_numpy().reshape(-1, 1).astype(float)

raw_train_df.shape, raw_test_df.shape


((957, 13), (240, 13))

## 15. Manual Missing-Value Imputation

Numeric columns: fill with the **training-set median**. Categorical columns: fill with the **training-set mode**. Statistics are learned only from the training split and reused on the test split to avoid leakage — the same principle Scikit-learn's  SimpleImputer  follows internally.


In [16]:
train_medians = raw_train_df[numeric_cols].median()
train_modes = raw_train_df[categorical_cols].mode().iloc[0]

train_filled = raw_train_df.copy()
test_filled = raw_test_df.copy()

train_filled[numeric_cols] = train_filled[numeric_cols].fillna(train_medians)
test_filled[numeric_cols] = test_filled[numeric_cols].fillna(train_medians)

train_filled[categorical_cols] = train_filled[categorical_cols].fillna(train_modes)
test_filled[categorical_cols] = test_filled[categorical_cols].fillna(train_modes)

print('Remaining missing values (train):', train_filled.isnull().sum().sum())
print('Remaining missing values (test): ', test_filled.isnull().sum().sum())


Remaining missing values (train): 0
Remaining missing values (test):  0


## 16. Manual Categorical Encoding (One-Hot)

 pd.get_dummies  builds the one-hot columns from the **training** categories; the test set is then re-indexed onto that exact same column set (any unseen category simply gets all-zero indicators, and any missing column is filled with 0) — equivalent to  OneHotEncoder(handle_unknown='ignore') .


In [17]:
train_cat_encoded = pd.get_dummies(train_filled[categorical_cols], columns=categorical_cols)
test_cat_encoded = pd.get_dummies(test_filled[categorical_cols], columns=categorical_cols)
test_cat_encoded = test_cat_encoded.reindex(columns=train_cat_encoded.columns, fill_value=0)

train_cat_encoded = train_cat_encoded.astype(float)
test_cat_encoded = test_cat_encoded.astype(float)

print('Encoded categorical shape (train/test):', train_cat_encoded.shape, test_cat_encoded.shape)


Encoded categorical shape (train/test): (957, 13) (240, 13)


## 17. Manual Feature Scaling (Standardization)

Z-score standardization: $z = \dfrac{x - \mu_{train}}{\sigma_{train}}$, computed with NumPy and applying the **training** mean/std to both splits.


In [ ]:
train_num = train_filled[numeric_cols].to_numpy(dtype=float)
test_num = test_filled[numeric_cols].to_numpy(dtype=float)

num_mean = train_num.mean(axis=0)
num_std = train_num.std(axis=0)
num_std[num_std == 0] = 1.0  

train_num_scaled = (train_num - num_mean) / num_std
test_num_scaled = (test_num - num_mean) / num_std

train_num_scaled.shape, test_num_scaled.shape


((957, 10), (240, 10))

## 18. Assemble the Final Design Matrices

Concatenate the scaled numeric block with the one-hot categorical block, then prepend a bias (intercept) column of ones. The **same** feature matrix is used for both regression and classification (only the target vector differs), mirroring Part A.


In [19]:
X_train_features = np.hstack([train_num_scaled, train_cat_encoded.to_numpy(dtype=float)])
X_test_features = np.hstack([test_num_scaled, test_cat_encoded.to_numpy(dtype=float)])

ones_train = np.ones((X_train_features.shape[0], 1))
ones_test = np.ones((X_test_features.shape[0], 1))

X_train_manual = np.hstack([ones_train, X_train_features])  # column 0 = bias/intercept
X_test_manual = np.hstack([ones_test, X_test_features])

print('Final design matrix shape (train/test):', X_train_manual.shape, X_test_manual.shape)


Final design matrix shape (train/test): (957, 24) (240, 24)


## 19. Manual Linear Regression — Closed-Form Normal Equation

$\theta = (X^T X)^{-1} X^T y$, solved with  np.linalg.pinv  (Moore-Penrose pseudo-inverse) for numerical stability against collinear/one-hot columns.


In [20]:
start = time.perf_counter()
XtX = X_train_manual.T @ X_train_manual
Xty = X_train_manual.T @ y_reg_train_manual
theta_reg = np.linalg.pinv(XtX) @ Xty
manual_reg_train_time = time.perf_counter() - start

start = time.perf_counter()
y_reg_pred_manual = X_test_manual @ theta_reg
manual_reg_predict_time = time.perf_counter() - start

print(f'Training time:   {manual_reg_train_time:.6f} s')
print(f'Prediction time: {manual_reg_predict_time:.6f} s')


Training time:   0.007564 s
Prediction time: 0.000052 s


## 20. Manual Regression Evaluation (MAE, RMSE, R²) — computed by hand with NumPy


In [21]:
errors = y_reg_test_manual - y_reg_pred_manual

manual_mae = np.mean(np.abs(errors))
manual_rmse = np.sqrt(np.mean(errors ** 2))

ss_res = np.sum(errors ** 2)
ss_tot = np.sum((y_reg_test_manual - np.mean(y_reg_test_manual)) ** 2)
manual_r2 = 1 - ss_res / ss_tot

print(f'MAE:  {manual_mae:.4f}')
print(f'RMSE: {manual_rmse:.4f}')
print(f'R2:   {manual_r2:.4f}')


MAE:  0.1088
RMSE: 0.1481
R2:   0.1736


## 21. Manual Logistic Regression — Sigmoid + Batch Gradient Descent

Implements the sigmoid function, binary cross-entropy cost, and full-batch **vectorized** gradient descent to learn the weights, then thresholds predicted probabilities at 0.5.


In [22]:
def sigmoid(z):
    z = np.clip(z, -500, 500)  # avoid overflow in exp
    return 1.0 / (1.0 + np.exp(-z))

def logistic_cost(X, y, w):
    m = X.shape[0]
    p = sigmoid(X @ w)
    eps = 1e-12
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

def train_logistic_gd(X, y, lr=0.1, n_iters=2000):
    m, n = X.shape
    w = np.zeros((n, 1))
    for i in range(n_iters):
        p = sigmoid(X @ w)
        grad = (X.T @ (p - y)) / m
        w -= lr * grad
    return w


In [23]:
start = time.perf_counter()
w_clf_gd = train_logistic_gd(X_train_manual, y_clf_train_manual, lr=0.5, n_iters=3000)
manual_clf_train_time = time.perf_counter() - start

start = time.perf_counter()
clf_probs_manual = sigmoid(X_test_manual @ w_clf_gd)
y_clf_pred_manual = (clf_probs_manual >= 0.5).astype(int)
manual_clf_predict_time = time.perf_counter() - start

print(f'Final training cost: {logistic_cost(X_train_manual, y_clf_train_manual, w_clf_gd):.4f}')
print(f'Training time:   {manual_clf_train_time:.6f} s')
print(f'Prediction time: {manual_clf_predict_time:.6f} s')


Final training cost: 0.4909
Training time:   0.064548 s
Prediction time: 0.000110 s


## 22. Manual Classification Evaluation — Confusion Matrix & Metrics by Hand


In [24]:
def confusion_counts(y_true, y_pred):
    y_true = y_true.ravel(); y_pred = y_pred.ravel()
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, tn, fp, fn

def classification_metrics(y_true, y_pred):
    tp, tn, fp, fn = confusion_counts(y_true, y_pred)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return accuracy, precision, recall, f1

manual_accuracy, manual_precision, manual_recall, manual_f1 = classification_metrics(y_clf_test_manual, y_clf_pred_manual)

print(f'Accuracy:  {manual_accuracy:.4f}')
print(f'Precision: {manual_precision:.4f}')
print(f'Recall:    {manual_recall:.4f}')
print(f'F1-score:  {manual_f1:.4f}')


Accuracy:  0.7583
Precision: 0.7767
Recall:    0.9435
F1-score:  0.8520


# Part C — Comparison and Optimization


## 23. Compact Comparison Tables — Scikit-learn vs. Manual (NumPy/Pandas)


In [25]:
regression_comparison = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R2', 'Train Time (s)', 'Predict Time (s)'],
    'Scikit-learn': [sklearn_mae, sklearn_rmse, sklearn_r2, sklearn_reg_train_time, sklearn_reg_predict_time],
    'Manual (NumPy/Pandas)': [manual_mae, manual_rmse, manual_r2, manual_reg_train_time, manual_reg_predict_time],
})
regression_comparison['Difference'] = regression_comparison['Scikit-learn'] - regression_comparison['Manual (NumPy/Pandas)']
print('Regression: Scikit-learn vs Manual')
regression_comparison


Regression: Scikit-learn vs Manual


,Metric,Scikit-learn,Manual (NumPy/Pandas),Difference
0,MAE,0.108771,0.108771,-2.081668e-16
1,RMSE,0.148134,0.148134,-8.326673e-17
2,R2,0.173576,0.173576,8.881784e-16
3,Train Time (s),0.033108,0.007564,2.554460e-02
4,Predict Time (s),0.007800,0.000052,7.747800e-03


In [26]:
classification_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'Train Time (s)', 'Predict Time (s)'],
    'Scikit-learn': [sklearn_accuracy, sklearn_precision, sklearn_recall, sklearn_f1, sklearn_clf_train_time, sklearn_clf_predict_time],
    'Manual (NumPy/Pandas)': [manual_accuracy, manual_precision, manual_recall, manual_f1, manual_clf_train_time, manual_clf_predict_time],
})
classification_comparison['Difference'] = classification_comparison['Scikit-learn'] - classification_comparison['Manual (NumPy/Pandas)']
print('Classification: Scikit-learn vs Manual')
classification_comparison


Classification: Scikit-learn vs Manual


,Metric,Scikit-learn,Manual (NumPy/Pandas),Difference
0,Accuracy,0.750000,0.758333,-0.008333
1,Precision,0.769585,0.776744,-0.007159
2,Recall,0.943503,0.943503,0.000000
3,F1-score,0.847716,0.852041,-0.004325
4,Train Time (s),0.030487,0.064548,-0.034062
5,Predict Time (s),0.008318,0.000110,0.008208


## 24. Optimizing the Manual Implementation

**Linear Regression** is already at its optimum: the closed-form normal equation *is* the exact least-squares solution, so there is no iterative accuracy to improve — only numerical stability (already handled via  pinv ).

**Logistic Regression** is where plain gradient descent can lag Scikit-learn's solver (LBFGS, a quasi-Newton method with fast, superlinear convergence). To close the gap, we re-implement training with **Newton's Method / Iteratively Reweighted Least Squares (IRLS)**:
- Uses the second-order Hessian, so it converges in far fewer iterations than gradient descent.
- Adds a small L2 (ridge) penalty on the weights (excluding the bias) for numerical stability when the Hessian is near-singular.
- Includes a convergence check (stops early once the gradient norm is small) instead of always running a fixed iteration budget.


In [27]:
def train_logistic_irls(X, y, n_iters=25, tol=1e-6, l2=1.0):
    m, n = X.shape
    w = np.zeros((n, 1))
    reg_mask = np.ones((n, 1)); reg_mask[0, 0] = 0.0  # don't regularize the bias term
    for i in range(n_iters):
        p = sigmoid(X @ w)
        grad = (X.T @ (p - y)) / m + l2 * (reg_mask * w) / m
        r = (p * (1 - p)).ravel()
        r = np.clip(r, 1e-6, None)  # avoid a singular Hessian
        # Hessian = (X^T diag(r) X)/m + L2 term, computed without building the full diag(r) matrix
        H = (X.T * r) @ X / m + l2 * np.diag(reg_mask.ravel()) / m
        delta = np.linalg.solve(H, grad)
        w -= delta
        if np.linalg.norm(delta) < tol:
            break
    return w, i + 1


In [28]:
start = time.perf_counter()
w_clf_irls, n_iters_used = train_logistic_irls(X_train_manual, y_clf_train_manual)
manual_opt_clf_train_time = time.perf_counter() - start

start = time.perf_counter()
clf_probs_opt = sigmoid(X_test_manual @ w_clf_irls)
y_clf_pred_opt = (clf_probs_opt >= 0.5).astype(int)
manual_opt_clf_predict_time = time.perf_counter() - start

manual_opt_accuracy, manual_opt_precision, manual_opt_recall, manual_opt_f1 = classification_metrics(y_clf_test_manual, y_clf_pred_opt)

print(f'Converged in {n_iters_used} Newton/IRLS iterations (vs. 3000 gradient-descent iterations)')
print(f'Training time:   {manual_opt_clf_train_time:.6f} s')
print(f'Prediction time: {manual_opt_clf_predict_time:.6f} s')
print(f'Accuracy:  {manual_opt_accuracy:.4f}')
print(f'Precision: {manual_opt_precision:.4f}')
print(f'Recall:    {manual_opt_recall:.4f}')
print(f'F1-score:  {manual_opt_f1:.4f}')


Converged in 8 Newton/IRLS iterations (vs. 3000 gradient-descent iterations)
Training time:   0.004166 s
Prediction time: 0.000123 s
Accuracy:  0.7500
Precision: 0.7696
Recall:    0.9435
F1-score:  0.8477


## 25. Final Three-Way Comparison — Scikit-learn vs. Manual GD vs. Manual Optimized (IRLS)


In [29]:
final_classification_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'Train Time (s)', 'Predict Time (s)'],
    'Scikit-learn': [sklearn_accuracy, sklearn_precision, sklearn_recall, sklearn_f1, sklearn_clf_train_time, sklearn_clf_predict_time],
    'Manual - Gradient Descent': [manual_accuracy, manual_precision, manual_recall, manual_f1, manual_clf_train_time, manual_clf_predict_time],
    'Manual - Optimized (IRLS)': [manual_opt_accuracy, manual_opt_precision, manual_opt_recall, manual_opt_f1, manual_opt_clf_train_time, manual_opt_clf_predict_time],
})
final_classification_comparison


,Metric,Scikit-learn,Manual - Gradient Descent,Manual - Optimized (IRLS)
0,Accuracy,0.750000,0.758333,0.750000
1,Precision,0.769585,0.776744,0.769585
2,Recall,0.943503,0.943503,0.943503
3,F1-score,0.847716,0.852041,0.847716
4,Train Time (s),0.030487,0.064548,0.004166
5,Predict Time (s),0.008318,0.000110,0.000123


## 26. Discussion — Why Performance differ

**Predictive performance:**
- Linear Regression: the manual closed-form solution and Scikit-learn's  LinearRegression  (which also solves least squares via an SVD-based routine) should match almost exactly — small differences come only from floating-point precision and the pseudo-inverse vs. SVD/ lstsq  internals, not from any algorithmic gap.
- Logistic Regression: plain gradient descent can under-fit slightly if it hasn't fully converged in the fixed iteration budget, since it only uses first-order (gradient) information. The Newton/IRLS version uses second-order curvature (the Hessian), converging to essentially the same optimum as Scikit-learn's LBFGS solver in a handful of iterations, which is why its metrics move much closer to the Scikit-learn results.